# CASA form extraction test (PaddleOCR-VL + RapidOCR)

Run on a free GPU (Colab: Runtime > Change runtime type > T4 GPU; Kaggle: Settings > Accelerator > GPU).

Upload CASA form JPG/PNG files in the upload cell. Optionally add `expected.json` mapping file name to the correct values, for example `{"form1.jpg": {"name": "MR SHANE SANCHEZ", "pan": "BRMPK0627D"}}`, to get a field-level comparison.

The results are written to `results.json` and `results.csv`, which you can download.

In [ ]:
# Do NOT upgrade torch or numpy here: upgrading them breaks the preinstalled torch on Colab/Kaggle.
!pip install -q rapidocr_onnxruntime accelerate
import transformers
print('transformers', transformers.__version__)
# PaddleOCR-VL needs transformers 5.x or newer. If the version above is older, run
# !pip install -q "transformers>=5"  then Runtime > Restart session, and run this cell again.
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import os, glob
IMAGE_DIR = '/content/casa_forms'
os.makedirs(IMAGE_DIR, exist_ok=True)
try:
    from google.colab import files  # Colab upload widget
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(IMAGE_DIR, name), 'wb') as f:
            f.write(data)
except ImportError:
    # Kaggle: attach a dataset with your images; they appear under /kaggle/input
    IMAGE_DIR = '/kaggle/input'
images = sorted(p for ext in ('*.jpg', '*.jpeg', '*.png') for p in glob.glob(os.path.join(IMAGE_DIR, '**', ext), recursive=True))
print(len(images), 'images:', [os.path.basename(p) for p in images])

In [ ]:
import time, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from rapidocr_onnxruntime import RapidOCR

t = time.time()
vl_path = 'PaddlePaddle/PaddleOCR-VL'
vl_proc = AutoProcessor.from_pretrained(vl_path)
vl_model = AutoModelForImageTextToText.from_pretrained(vl_path, dtype=torch.float16).cuda().eval()
rapid = RapidOCR()
print('models loaded in', round(time.time() - t, 1), 's')

In [ ]:
import json, csv, cv2
results = {}
for path in images:
    name = os.path.basename(path)
    img = Image.open(path).convert('RGB')

    # PaddleOCR-VL, whole page
    t = time.time()
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'OCR:'}]}]
    text = vl_proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = vl_proc(text=[text], images=[img], return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = vl_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    vl_text = vl_proc.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    vl_secs = round(time.time() - t, 2)

    # RapidOCR, printed text (CPU)
    t = time.time()
    ocr_res, _ = rapid(cv2.imread(path))
    rapid_text = '\n'.join(r[1] for r in (ocr_res or []))
    rapid_secs = round(time.time() - t, 2)

    results[name] = {'paddleocr_vl_text': vl_text, 'paddleocr_vl_secs': vl_secs,
                     'rapidocr_text': rapid_text, 'rapidocr_secs': rapid_secs}
    print(f'== {name}  VL {vl_secs}s  Rapid {rapid_secs}s')
    print(vl_text[:600])
    print()

In [ ]:
# Optional: field-level check against expected.json (if you uploaded it)
expected_path = os.path.join(IMAGE_DIR, 'expected.json')
if os.path.exists(expected_path):
    expected = json.load(open(expected_path))
    norm = lambda s: ''.join(ch for ch in s.upper() if ch.isalnum())
    for name, fields in expected.items():
        if name not in results:
            continue
        text_norm = norm(results[name]['paddleocr_vl_text'])
        for field, value in fields.items():
            hit = norm(value) in text_norm
            print(f'{name:20s} {field:12s} expected={value!r:28s} found_in_vl_text={hit}')
else:
    print('No expected.json uploaded: skipping field comparison.')

In [ ]:
json.dump(results, open('results.json', 'w'), indent=2, ensure_ascii=False)
with open('results.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['file', 'paddleocr_vl_secs', 'rapidocr_secs'])
    for name, r in results.items():
        w.writerow([name, r['paddleocr_vl_secs'], r['rapidocr_secs']])
print('saved results.json and results.csv')